In [150]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [215]:
from priorbox import AnchorBoxes
import yaml
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner

with open('config/priorbox.yaml', 'r') as file:
        config = yaml.safe_load(file)

boxes=AnchorBoxes(config)
anchors=boxes.forward()
gt_boxes = [
    normalised_gt_coords(75, 80, 60, 90,300,300),
    normalised_gt_coords(220, 70, 50, 40,300,300),
    normalised_gt_coords(150, 170, 140, 110,300,300),
    normalised_gt_coords(260, 240, 70, 120,300,300),
    normalised_gt_coords(45, 230, 80, 60,300,300)
]
gtboxes=torch.tensor(gt_boxes, dtype=torch.float32)
gtboxes

gt={}

for i,el in enumerate(gtboxes):
        
        gt[i]={"gtbox":tuple(el.tolist()),"class": 1 if i%2==0 else 2}

In [225]:
gt

{0: {'gtbox': (0.25,
   0.2666666805744171,
   0.20000000298023224,
   0.30000001192092896),
  'class': 1},
 1: {'gtbox': (0.7333333492279053,
   0.23333333432674408,
   0.1666666716337204,
   0.13333334028720856),
  'class': 2},
 2: {'gtbox': (0.5,
   0.5666666626930237,
   0.46666666865348816,
   0.36666667461395264),
  'class': 1},
 3: {'gtbox': (0.8666666746139526,
   0.800000011920929,
   0.23333333432674408,
   0.4000000059604645),
  'class': 2},
 4: {'gtbox': (0.15000000596046448,
   0.7666666507720947,
   0.2666666805744171,
   0.20000000298023224),
  'class': 1}}

In [229]:
best_matching={}
bests=set()
for gt_box in gt.keys():
        best=0
        best_anchor=None
        for ida,anchor in enumerate(anchors):

            iou_val=iou(*center_to_corner(*anchor),*center_to_corner(*gt[gt_box]["gtbox"]))
            if iou_val>best:
                  best=iou_val
                  best_anchor=ida

        best_matching[best_anchor]={"gtbox":gt_box,"iou":best,"class": gt[gt_box]["class"]}

In [232]:
""" keys are anchors"""
best_matching


{389: {'gtbox': 0, 'iou': tensor(0.6876), 'class': 1},
 1813: {'gtbox': 1, 'iou': tensor(0.5556), 'class': 2},
 5975: {'gtbox': 2, 'iou': tensor(0.7956), 'class': 1},
 7160: {'gtbox': 3, 'iou': tensor(0.9143), 'class': 2},
 7486: {'gtbox': 4, 'iou': tensor(0.8060), 'class': 1}}

In [233]:
any_matching={}
for id,anchor in enumerate(anchors):
      
      inner={}
      i=0
      best_iou=0
      best_gt=None
      best_class=None
      for gt_box in gt.keys():
            iou_val=iou(*center_to_corner(*anchor),*center_to_corner(*gt[gt_box]["gtbox"]))
            if iou_val>=0.5:
                  inner[i]={"gtbox":gt_box,"iou":iou_val}
                  i=i+1
            
            if iou_val>best_iou:
                 best_iou=iou_val
                 best_gt=gt_box
                 best_class=gt[gt_box]["class"]

      if inner:
        any_matching[id]={"class" :best_class,"best_gt" : best_gt ,"data":inner}
      elif id in best_matching:
           any_matching[id]={"class" :best_matching[id]["class"],"best_gt" : best_matching[id]["gtbox"] ,"data":inner}
           
           
      else:
           any_matching[id]={"class" :0,"best_gt" : None ,"data":None}
           

In [234]:
any_matching

{0: {'class': 0, 'best_gt': None, 'data': None},
 1: {'class': 0, 'best_gt': None, 'data': None},
 2: {'class': 0, 'best_gt': None, 'data': None},
 3: {'class': 0, 'best_gt': None, 'data': None},
 4: {'class': 0, 'best_gt': None, 'data': None},
 5: {'class': 0, 'best_gt': None, 'data': None},
 6: {'class': 0, 'best_gt': None, 'data': None},
 7: {'class': 0, 'best_gt': None, 'data': None},
 8: {'class': 0, 'best_gt': None, 'data': None},
 9: {'class': 0, 'best_gt': None, 'data': None},
 10: {'class': 0, 'best_gt': None, 'data': None},
 11: {'class': 0, 'best_gt': None, 'data': None},
 12: {'class': 0, 'best_gt': None, 'data': None},
 13: {'class': 0, 'best_gt': None, 'data': None},
 14: {'class': 0, 'best_gt': None, 'data': None},
 15: {'class': 0, 'best_gt': None, 'data': None},
 16: {'class': 0, 'best_gt': None, 'data': None},
 17: {'class': 0, 'best_gt': None, 'data': None},
 18: {'class': 0, 'best_gt': None, 'data': None},
 19: {'class': 0, 'best_gt': None, 'data': None},
 20: {'cla

In [147]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

In [148]:
base=nn.ModuleList(vgg[:30])#until 5_3 layer

In [149]:
base
convs={}
i=2
convid=1
print("----","conv",1,"----")

for id, el in enumerate(base):

    print(f"({id})",el )
    if isinstance(el,nn.Conv2d):
        
        convs[id]=f"{i}_{convid}"
        convid=convid+1
    if isinstance(el,nn.MaxPool2d):
        convid=1
        print("----","conv",i,"----")
        i=i+1
    

    

    
    

---- conv 1 ----
(0) Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(1) ReLU(inplace=True)
(2) Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(3) ReLU(inplace=True)
(4) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 2 ----
(5) Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(6) ReLU(inplace=True)
(7) Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(8) ReLU(inplace=True)
(9) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 3 ----
(10) Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(11) ReLU(inplace=True)
(12) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(13) ReLU(inplace=True)
(14) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(15) ReLU(inplace=True)
(16) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
---- conv 4 ----
(17) Conv2d(256, 512, kernel_s

In [ ]:
from l2norm import L2norm
class SSD(nn.Module):
    def __init__(self,base,nb_classes):
        super().__init__()
        self.features=base
        self.nb_classes=nb_classes

        self.l2norm=L2norm(512,20)

        self.extras=nn.ModuleList([
            #conv6 and conv7
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,padding=6,dilation=6),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=1),
                nn.ReLU(inplace=True),    
            ),

            #conv8_2
            nn.Sequential(
                nn.Conv2d(in_channels=1024,out_channels=256,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv9_2
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv10_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            ),

            #conv11_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            )
        ])

        #define kernels for classification to output Feature Map of sieze H,W,ki*nb_classes for all H,W, ki in {4,6} for all i =1 ... |ssd feature maps | , ki is number of anchors for each location of H,W, image 
        self.classification_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(512,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
        ])

        #same but using 4 coordinates for each anchor 
        self.regression_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*4,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*4,kernel_size=3,padding=1),
            nn.Conv2d(512,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),

        ])
    def forward(self, X):
        layers_for_prediction = []
      
        #base model 
        for idx in range(len(self.features)):

            X=self.features[idx](X)
            
            if idx in convs and convs[idx]=="4_3":

                X=self.l2norm(X)
                layers_for_prediction.append(X)
          
                
        for idx in range(len(self.extras)):
            X=self.extras[idx](X)
            
            layers_for_prediction.append(X)


        classifications=[]    
        for layer_for_predictions, classification_convolution in zip( layers_for_prediction, self.classification_convolutions):
            classifications.append(classification_convolution(layer_for_predictions))
        
        regressions=[]  
        for layer_for_predictions, regression_convolution in zip( layers_for_prediction, self.regression_convolutions):
            regressions.append(regression_convolution(layer_for_predictions))

        return classifications,regressions,layers_for_prediction



In [38]:
convs

{0: '1_1',
 2: '1_2',
 5: '2_1',
 7: '2_2',
 10: '3_1',
 12: '3_2',
 14: '3_3',
 17: '4_1',
 19: '4_2',
 21: '4_3',
 24: '5_1',
 26: '5_2',
 28: '5_3'}

In [34]:
layers=[]
pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
layers += [pool5, conv6,
               nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
layers[1]

Conv2d(512, 1024, kernel_size=(3, 3), stride=(1, 1), padding=(6, 6), dilation=(6, 6))

In [29]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
classifications,regressions,layers_for_prediction=model.forward(X)

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
